# Create sliding-window variant data for TERT promoter mutations

Slide a 1 Mb window (step = 10 bp) so the variant appears at different positions
near the left edge of the window. Done separately for C228T and C250T.

Coordinates (hg38, 1-based, + strand; TERT is on the minus strand):

| Feature | hg19       | hg38       |
|---------|------------|------------|
| TSS     | 1,295,162  | 1,295,047  |
| ATG     | 1,295,104  | 1,294,989  |
| C228T   | 1,295,228  | 1,295,113  |
| C250T   | 1,295,250  | 1,295,135  |

hg19→hg38 shift: −115 bp. Distances (−124/−146) are from the ATG, not the TSS.

Window start ranges from **(TSS + 20)** to **variant_pos** in steps of 20 bp.  
TSS stays outside / at least 20 bp left of the window left edge.

In [1]:
import pandas as pd
import os

# hg38, 1-based, + strand
TSS = 1_295_047

mutations = {
    'C228T': {'chrom': 'chr5', 'pos': 1_295_113, 'ref': 'G', 'alt': 'A'},
    'C250T': {'chrom': 'chr5', 'pos': 1_295_135, 'ref': 'G', 'alt': 'A'},
}

WINDOW_SIZE = 1_000_000
SLIDE       = 10          # step size in bp
TSS_MARGIN  = 20          # window start must be >= TSS + TSS_MARGIN

OUTPUT_DIR = '/scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
def make_windows(chrom, pos, ref, alt, tss=TSS,
                 window_size=WINDOW_SIZE, slide=SLIDE, tss_margin=TSS_MARGIN):
    """
    Window start slides from (tss + tss_margin) to pos in steps of slide bp.
    The variant at pos is always inside the window; TSS stays left of the window
    by at least tss_margin bp.
    """
    min_start = tss + tss_margin   # left-most window start (TSS just outside)
    max_start = pos                # right-most start (variant at window position 0)

    records = []
    for ws in range(min_start, max_start + 1, slide):
        records.append({
            'chrom'            : chrom,
            'window_start'     : ws,                  # 1-based inclusive
            'window_end'       : ws + window_size - 1,
            'var_pos'          : pos,                 # 1-based
            'var_pos_in_window': pos - ws,            # 0-based offset from left edge
            'ref'              : ref,
            'alt'              : alt,
        })
    return pd.DataFrame(records)


for name, mut in mutations.items():
    df = make_windows(**mut)
    print(f"{name}: {len(df)} windows, "
          f"var_pos_in_window from {df['var_pos_in_window'].min()} "
          f"to {df['var_pos_in_window'].max()}")
    print(df.head())
    print()

C228T: 5 windows, var_pos_in_window from 6 to 46
  chrom  window_start  window_end  var_pos  var_pos_in_window ref alt
0  chr5       1295067     2295066  1295113                 46   G   A
1  chr5       1295077     2295076  1295113                 36   G   A
2  chr5       1295087     2295086  1295113                 26   G   A
3  chr5       1295097     2295096  1295113                 16   G   A
4  chr5       1295107     2295106  1295113                  6   G   A

C250T: 7 windows, var_pos_in_window from 8 to 68
  chrom  window_start  window_end  var_pos  var_pos_in_window ref alt
0  chr5       1295067     2295066  1295135                 68   G   A
1  chr5       1295077     2295076  1295135                 58   G   A
2  chr5       1295087     2295086  1295135                 48   G   A
3  chr5       1295097     2295096  1295135                 38   G   A
4  chr5       1295107     2295106  1295135                 28   G   A



In [3]:
# Save each mutation's windows separately
for name, mut in mutations.items():
    df = make_windows(**mut)
    out_path = os.path.join(OUTPUT_DIR, f'tert_{name}_sliding_windows.tsv')
    df.to_csv(out_path, sep='\t', index=False)
    print(f"Saved {len(df)} windows → {out_path}")

Saved 5 windows → /scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/tert_C228T_sliding_windows.tsv
Saved 7 windows → /scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/tert_C250T_sliding_windows.tsv
